# JAX PSD boundaries and original CPU validation

This notebook fixes input bytes, dtype, segmentation and grids while comparing ordinary CPU, default JAX and selected original CPU routes. It asserts exact CPU/native dtype, shape, metadata and bytes. Default JAX remains on the selected device; each native control replaces only its named operation.

Controlled window, transform, power, reduction and division examples explain numerical boundaries without attributing every residual to an unobserved compiler detail. The examples cover Welch averaging, inverse-ASD/PSD truncation, interpolation and one analytical model. Recorded results apply to the versions/device below and do not qualify a complete search. Install PyCBC, JAX and Jupyter in the same environment.

For a **self-contained NumPy/JAX demonstration** without PyCBC or LAL, run
[psd rounding](jax_psd_rounding.ipynb). That notebook illustrates arithmetic
with generated inputs. This notebook checks the actual PyCBC implementations
and original validation routes. Results apply to each notebook's recorded
libraries and device.


In [1]:
import platform
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as np
import scipy
import jax
import jax.numpy as jnp
import jaxlib
import pycbc
import lal
import lalsimulation
from pycbc.scheme import CPUScheme, JAXScheme
from pycbc.types import TimeSeries, FrequencySeries
from pycbc.fft import fft, backend_cpu
from pycbc.psd import (welch, welch_jax, interpolate, inverse_spectrum_truncation,
                       from_string, median_bias)

DEVICE = "cpu"  # Use "cuda:0" to repeat on an available NVIDIA GPU.
with JAXScheme(DEVICE) as context:
    metadata = {
        "Python": platform.python_version(), "CPU architecture": platform.machine(),
        "PyCBC": pycbc.__version__, "NumPy": np.__version__,
        "SciPy": scipy.__version__, "JAX": jax.__version__,
        "jaxlib": jaxlib.__version__, "LAL": lal.__version__,
        "LALSimulation": lalsimulation.__version__,
        "CPU FFT backend": backend_cpu.get_backend().__name__,
        "JAX platform": context.jax_device.platform,
        "JAX device kind": context.jax_device.device_kind,
        "JAX x64": jax.config.jax_enable_x64,
    }
assert metadata["JAX x64"], "These double-precision controls require JAX x64."
for name, value in metadata.items():
    print(f"{name}: {value}")
rng = np.random.default_rng(1729)
samples = rng.standard_normal(256)
psd_values = 1 + 0.05 * np.arange(33) + 0.2 * np.sin(np.arange(33) / 3)**2


Python: 3.12.7
CPU architecture: arm64
PyCBC: 0.0a9304
NumPy: 2.5.2
SciPy: 1.16.3
JAX: 0.11.1
jaxlib: 0.11.1
LAL: 7.7.1
LALSimulation: 6.2.1
CPU FFT backend: pycbc.fft.fftw
JAX platform: cpu
JAX device kind: cpu
JAX x64: True


## Exact native comparisons on fixed inputs

Whole-stage native controls invoke the original CPU calculation in a separate process. Each call below gets fresh arrays. Epoch and spacing are checked alongside output bytes. This takes longer than default execution because process startup and host transfers are part of the validation route.

In [2]:
def snapshot(series):
    return {"values": np.array(series.numpy(), copy=True),
            "delta_f": series.delta_f,
            "epoch": None if series.epoch is None else float(series.epoch)}


def evaluate(operation, values=None, *, cpu=False, reference=(), **options):
    context = CPUScheme() if cpu else JAXScheme(
        DEVICE, reference_operations=reference)
    with context:
        if operation == "welch":
            source = TimeSeries(values.copy(), delta_t=1 / 256, epoch=1000000000)
            result = welch(source, seg_len=64, seg_stride=32, num_segments=7,
                           require_exact_data_fit=True, **options)
        elif operation in ("interpolate", "inverse_spectrum_truncation"):
            source = FrequencySeries(values.copy(), delta_f=4, epoch=1000000000)
            result = (interpolate if operation == "interpolate"
                      else inverse_spectrum_truncation)(source, **options)
        elif operation == "analytical_psd":
            result = from_string(**options)
        else:
            raise ValueError(operation)
        return snapshot(result)


def assert_exact(actual, expected):
    a, b = actual["values"], expected["values"]
    assert a.dtype == b.dtype and a.shape == b.shape
    assert a.tobytes() == b.tobytes()
    assert actual["delta_f"] == expected["delta_f"]
    assert actual["epoch"] == expected["epoch"]


def report(name, cpu, candidate):
    a, b = cpu["values"], candidate["values"]
    finite = np.isfinite(a) & np.isfinite(b)
    error = float(np.max(np.abs(a[finite] - b[finite]))) if np.any(finite) else 0
    scale = max(float(np.max(np.abs(a[finite]))), np.finfo(a.dtype).tiny)
    print(f"{name}: CPU dtype={a.dtype}, JAX dtype={b.dtype}; "
          f"max abs={error:.6g}; scaled max={error / scale:.6g}; "
          f"value mismatches={np.count_nonzero(a != b)}/{a.size}")


checks = {}
def compare(name, operation, values=None, **options):
    cpu = evaluate(operation, values, cpu=True, **options)
    default = evaluate(operation, values, **options)
    native = evaluate(operation, values, reference=(operation,), **options)
    assert_exact(native, cpu)
    report(name, cpu, default)
    print("  native dtype/shape/metadata/bytes: exact")
    checks[name] = (cpu, default, native)
    return cpu, default, native


for dtype in (np.float32, np.float64):
    for averaging in ("mean", "median", "median-mean"):
        _ = compare(f"{dtype.__name__}/{averaging}", "welch",
                    samples.astype(dtype), avg_method=averaging)


float32/mean: CPU dtype=float32, JAX dtype=float32; max abs=2.32831e-09; scaled max=2.4558e-07; value mismatches=17/33
  native dtype/shape/metadata/bytes: exact


float32/median: CPU dtype=float32, JAX dtype=float32; max abs=2.79397e-09; scaled max=2.18244e-07; value mismatches=14/33
  native dtype/shape/metadata/bytes: exact


float32/median-mean: CPU dtype=float32, JAX dtype=float32; max abs=1.39698e-09; scaled max=1.03567e-07; value mismatches=14/33
  native dtype/shape/metadata/bytes: exact


float64/mean: CPU dtype=float64, JAX dtype=float64; max abs=3.46945e-18; scaled max=3.65942e-16; value mismatches=23/33
  native dtype/shape/metadata/bytes: exact


float64/median: CPU dtype=float64, JAX dtype=float64; max abs=6.93889e-18; scaled max=5.42016e-16; value mismatches=26/33
  native dtype/shape/metadata/bytes: exact


float64/median-mean: CPU dtype=float64, JAX dtype=float64; max abs=6.93889e-18; scaled max=5.14421e-16; value mismatches=27/33
  native dtype/shape/metadata/bytes: exact


## Window and transform isolation

NumPy and JAX Hann construction can round differently. Casting to float32 may remove a difference visible in float64. The window-energy sum is another boundary. Supplying one common explicit window fixes window values; selecting `fft` replaces the internal segment transforms while retaining JAX power, averaging and normalization. Neither control replaces all Welch arithmetic.

In [3]:
for dtype in (np.float32, np.float64):
    common = np.hanning(64).astype(dtype)
    with JAXScheme(DEVICE):
        generated = np.asarray(jnp.hanning(64).astype(dtype))
        energy_jax = np.asarray(jnp.sum(jnp.square(jnp.asarray(common))))
    print(dtype.__name__, "Hann value mismatches:", np.count_nonzero(common != generated),
          "; max window error:", np.max(np.abs(common - generated)))
    print("  common-window energy CPU/JAX:", np.sum(common * common), energy_jax)
    cpu = evaluate("welch", samples.astype(dtype), cpu=True,
                   avg_method="mean", window=common)
    common_jax = evaluate("welch", samples.astype(dtype),
                          avg_method="mean", window=common)
    original_fft = evaluate("welch", samples.astype(dtype), reference=("fft",),
                            avg_method="mean", window=common)
    report(dtype.__name__ + "/common window", cpu, common_jax)
    report(dtype.__name__ + "/common window + original FFT", cpu, original_fft)


float32 Hann value mismatches: 0 ; max window error: 0.0
  common-window energy CPU/JAX: 23.625 23.625
float32/common window: CPU dtype=float32, JAX dtype=float32; max abs=2.32831e-09; scaled max=2.4558e-07; value mismatches=17/33
float32/common window + original FFT: CPU dtype=float32, JAX dtype=float32; max abs=9.31323e-10; scaled max=9.82318e-08; value mismatches=11/33
float64 Hann value mismatches: 44 ; max window error: 3.885780586188048e-16
  common-window energy CPU/JAX: 23.625 23.624999999999996
float64/common window: CPU dtype=float64, JAX dtype=float64; max abs=2.60209e-18; scaled max=2.74457e-16; value mismatches=21/33
float64/common window + original FFT: CPU dtype=float64, JAX dtype=float64; max abs=1.73472e-18; scaled max=1.82971e-16; value mismatches=20/33


## Frozen-spectrum power and averaging controls

The CPU Welch implementation takes the magnitude of `spectrum * conjugate(spectrum)`; JAX takes its real part. These agree in exact arithmetic. The following freezes a CPU spectrum first, so any power-expression difference cannot originate in the FFT input.

A separate fixed periodogram matrix isolates mean/median and division from transforms. NumPy scalar division is compared with widened division rounded back to float32 and with multiplication by a rounded reciprocal. The latter is a control for a different rounding sequence, not a claim that every JAX device uses it.

In [4]:
with CPUScheme():
    segment = TimeSeries((samples[:64].astype(np.float32)
                          * np.hanning(64).astype(np.float32)), delta_t=1 / 256)
    spectrum = FrequencySeries(np.zeros(33, dtype=np.complex64), delta_f=4)
    fft(segment, spectrum)
    fixed_spectrum = np.array(spectrum.numpy(), copy=True)
    cpu_power = np.array(abs(spectrum * spectrum.conj()).numpy(), copy=True)
with JAXScheme(DEVICE):
    jax_power = np.asarray(jax.jit(lambda z: jnp.real(z * jnp.conj(z)))(
        jnp.asarray(fixed_spectrum)))
print("Fixed-spectrum power maximum error:", np.max(np.abs(cpu_power - jax_power)),
      "; value mismatches:", np.count_nonzero(cpu_power != jax_power))

fixed = (0.3 + np.abs(rng.standard_normal((7, 9)))).astype(np.float32)
with JAXScheme(DEVICE):
    fixed_jax = jnp.asarray(fixed)
    for name in ("mean", "median"):
        cpu = getattr(np, name)(fixed, axis=0)
        result = np.asarray(getattr(jnp, name)(fixed_jax, axis=0))
        print("Fixed", name, "maximum error:", np.max(np.abs(cpu - result)))
    bias = np.float32(median_bias(7))
    numerator = np.median(fixed, axis=0)
    cpu_division = numerator / bias
    wide_division = np.asarray((jnp.asarray(numerator).astype(jnp.float64)
                               / jnp.asarray(bias, dtype=jnp.float64)).astype(jnp.float32))
    reciprocal_multiply = numerator * np.float32(1 / bias)
np.testing.assert_array_equal(wide_division, cpu_division)
print("Median bias:", bias, "; widened division matches CPU for these fixed inputs.")
print("Rounded-reciprocal multiplication differences:",
      np.count_nonzero(reciprocal_multiply != cpu_division),
      "; max error:", np.max(np.abs(reciprocal_multiply - cpu_division)))


Fixed-spectrum power maximum error: 0.0 ; value mismatches: 0
Fixed mean maximum error: 1.1920929e-07
Fixed median maximum error: 0.0
Median bias: 0.7595238 ; widened division matches CPU for these fixed inputs.
Rounded-reciprocal multiplication differences: 3 ; max error: 2.3841858e-07


## Explicit wider precision

Normal Welch estimation retains input precision. `wide_fft=True` is an explicit lower-level option: window/sample multiplication remains float32, then the FFT and PSD arithmetic use float64. It is a different numerical policy, not an original CPU route.

In [5]:
with JAXScheme(DEVICE):
    source = TimeSeries(samples.astype(np.float32), delta_t=1 / 256, epoch=1000000000)
    wider = snapshot(welch_jax(source, seg_len=64, seg_stride=32, num_segments=7,
                              require_exact_data_fit=True, avg_method="mean", wide_fft=True))
assert wider["values"].dtype == np.float64
report("explicit wide_fft", checks["float32/mean"][0], wider)


explicit wide_fft: CPU dtype=float32, JAX dtype=float64; max abs=2.16018e-09; scaled max=2.27846e-07; value mismatches=33/33


## Inverse-ASD and inverse-PSD truncation

Inverse ASD adds a square root before transforming to time; inverse PSD omits it. Both truncate/taper the response, transform back and take a reciprocal. Reciprocal/square-root precision, windows, transform scaling and final power/magnitude arithmetic are separate boundaries. JAX normalizes the inverse transform by `1/N`; the original functional pair scales by `delta_f` and then `1/(N*delta_f)`. Their net scaling agrees in exact arithmetic but places rounding differently.

Three native checks cover both spectrum choices, hard/Hann truncation and both input precisions. The transform-only controls keep surrounding JAX arithmetic selected, so residuals are reported without assuming original FFTs must make the full stage identical.

In [6]:
truncation_cases = [
    (np.float32, "invasd", "hann"),
    (np.float32, "invpsd", None),
    (np.float64, "invasd", None),
]
for dtype, spectrum_kind, taper in truncation_cases:
    _ = compare(f"{dtype.__name__}/{spectrum_kind}/{taper}",
                "inverse_spectrum_truncation", psd_values.astype(dtype),
                max_filter_len=16, which_spectrum=spectrum_kind,
                low_frequency_cutoff=8, trunc_method=taper)

cpu = checks["float32/invasd/hann"][0]
for operations in (("ifft",), ("fft",), ("fft", "ifft")):
    result = evaluate("inverse_spectrum_truncation", psd_values.astype(np.float32),
                      reference=operations, max_filter_len=16,
                      which_spectrum="invasd", low_frequency_cutoff=8,
                      trunc_method="hann")
    report("transform-only " + ",".join(operations), cpu, result)


float32/invasd/hann: CPU dtype=float32, JAX dtype=float32; max abs=7.15256e-07; scaled max=2.16079e-07; value mismatches=4/33
  native dtype/shape/metadata/bytes: exact


float32/invpsd/None: CPU dtype=float32, JAX dtype=float32; max abs=1.19209e-07; scaled max=3.30319e-08; value mismatches=1/33
  native dtype/shape/metadata/bytes: exact


float64/invasd/None: CPU dtype=float64, JAX dtype=float64; max abs=8.88178e-16; scaled max=7.47484e-17; value mismatches=4/33
  native dtype/shape/metadata/bytes: exact
transform-only ifft: CPU dtype=float32, JAX dtype=float32; max abs=7.15256e-07; scaled max=2.16079e-07; value mismatches=4/33
transform-only fft: CPU dtype=float32, JAX dtype=float32; max abs=7.15256e-07; scaled max=2.16079e-07; value mismatches=2/33
transform-only fft,ifft: CPU dtype=float32, JAX dtype=float32; max abs=7.15256e-07; scaled max=2.16079e-07; value mismatches=4/33


## Interpolation and an analytical model

Interpolation fixes the input grid at 4 Hz, the output spacing at 1.7 Hz and its length at 80 bins. Intermediate values are evaluated in double precision before returning to input storage precision; differing grid/arithmetic rounding can still affect results. Endpoint extension is checked explicitly.

The representative model uses a fractional cutoff. The series API follows the original cutoff-bin convention `int(low_freq_cutoff / delta_f)`. One model example does not validate every ported formula or table.

In [7]:
for dtype in (np.float32, np.float64):
    cpu, default, native = compare(dtype.__name__ + "/interpolation", "interpolate",
                                   psd_values.astype(dtype), delta_f=1.7, length=80)
    assert cpu["values"][-1] == psd_values.astype(dtype)[-1]
    assert default["values"][-1] == psd_values.astype(dtype)[-1]

cpu, default, native = compare("aLIGOZeroDetHighPower", "analytical_psd",
                               psd_name="aLIGOZeroDetHighPower", length=33,
                               delta_f=4, low_freq_cutoff=8.5)
expected_start = int(8.5 / 4)
assert np.flatnonzero(cpu["values"])[0] == expected_start
assert np.flatnonzero(default["values"])[0] == expected_start
assert len(checks) == 12
print("All 12 whole-stage native dtype/shape/metadata/byte assertions passed.")


float32/interpolation: CPU dtype=float32, JAX dtype=float32; max abs=0; scaled max=0; value mismatches=0/80
  native dtype/shape/metadata/bytes: exact


float64/interpolation: CPU dtype=float64, JAX dtype=float64; max abs=0; scaled max=0; value mismatches=0/80
  native dtype/shape/metadata/bytes: exact


aLIGOZeroDetHighPower: CPU dtype=float64, JAX dtype=float64; max abs=3.11151e-61; scaled max=1.17434e-17; value mismatches=9/33
  native dtype/shape/metadata/bytes: exact
All 12 whole-stage native dtype/shape/metadata/byte assertions passed.


## Physical constants and table boundaries

The [original quantum formula](https://lscsoft.docs.ligo.org/lalsuite/dev/lalsimulation/_l_a_l_sim_noise_p_s_d_8c_source.html#l00272) makes PSD proportional to reduced Planck's constant. A ratio control changes only that constant, preserving every other factor.

The [original table algorithm](https://lscsoft.docs.ligo.org/lalsuite/dev/lalsimulation/_l_a_l_sim_noise_p_s_d_8c_source.html#l01093) extrapolates log ASD from its selected interval. Compare that convention with clamping at the first row. These controls demonstrate mathematical choices, separately from remaining arithmetic rounding.

In [8]:
from pycbc.psd import analytical_psd_jax

hbar = 6.62607015e-34 / (2 * np.pi)
rounded_hbar = 1.054571817e-34
assert hbar == lal.HBAR_SI
frequencies = np.array([40., 100., 250.])
original_quantum = np.array([
    lalsimulation.SimNoisePSDaLIGOQuantumBHBH20Deg(float(f))
    for f in frequencies])
with JAXScheme(DEVICE):
    quantum = np.asarray(analytical_psd_jax(
        "aLIGOQuantumBHBH20Deg", jnp.asarray(frequencies)))
rounded_control = quantum * (rounded_hbar / hbar)
np.testing.assert_allclose(quantum, original_quantum, rtol=2e-13, atol=0)
relative_shift = (rounded_control - quantum) / quantum
np.testing.assert_allclose(relative_shift, rounded_hbar / hbar - 1,
                           rtol=1e-5, atol=1e-15)
print("hbar from h/(2*pi):", hbar)
print("Rounded-constant quantum PSD relative shifts:", relative_shift)
print("Default quantum/LAL maximum relative error:",
      np.max(np.abs(quantum / original_quantum - 1)))

filename = "LIGO-P1200087-v18-aLIGO_DESIGN.txt"
table_f, table_asd = np.loadtxt(lal.FileResolvePath(filename), unpack=True)
with CPUScheme():
    original_table = float(from_string(
        "aLIGODesignSensitivityP1200087", 33, 1, 9)[9])
with JAXScheme(DEVICE):
    device_table = float(from_string(
        "aLIGODesignSensitivityP1200087", 33, 1, 9)[9])
weight = (table_f[2] - 9) / (table_f[2] - table_f[1])
extrapolated = np.exp(2 * (weight * np.log(table_asd[1])
                          + (1 - weight) * np.log(table_asd[2])))
clamped = np.exp(2 * np.interp(9, table_f, np.log(table_asd)))
np.testing.assert_allclose(extrapolated, original_table, rtol=2e-13, atol=0)
np.testing.assert_allclose(device_table, original_table, rtol=2e-13, atol=0)
assert abs(clamped / original_table - 1) > 0.1
print("Versioned table:", filename)
print("First three frequency/ASD rows:", np.column_stack((table_f[:3], table_asd[:3])))
print("9 Hz PSD, original/JAX/log-ASD control:",
      original_table, device_table, extrapolated)
print("9 Hz clamped control:", clamped,
      "; relative difference:", clamped / original_table - 1)


hbar from h/(2*pi): 1.0545718176461565e-34
Rounded-constant quantum PSD relative shifts: [-6.12719372e-10 -6.12719365e-10 -6.12719309e-10]
Default quantum/LAL maximum relative error: 2.220446049250313e-16


Versioned table: LIGO-P1200087-v18-aLIGO_DESIGN.txt
First three frequency/ASD rows: [[9.0000000e+00 1.7370723e-21]
 [9.0203998e+00 2.1715963e-21]
 [9.0408458e+00 2.9166699e-21]]
9 Hz PSD, original/JAX/log-ASD control: 2.617705357617846e-42 2.617705357617846e-42 2.617705357617846e-42
9 Hz clamped control: 3.0174201754273013e-42 ; relative difference: 0.1526966419831155


## Selecting boundaries in a larger calculation

Use `JAXScheme(DEVICE, reference_operations=("welch",))` to replace only Welch, or choose `inverse_spectrum_truncation`, `interpolate` and `analytical_psd` independently. The CLI accepts comma-separated names through `--jax-reference-operations`. `fft` and `ifft` also control the corresponding internal PSD transforms without replacing surrounding arithmetic.

Whole-stage native routes use the original CPU implementation in a separate process and return arrays to the JAX device. Startup and transfers can make them much slower; JAX tracing and differentiation cannot cross that boundary. These exact assertions compare the same CPU implementation and input bytes, while the controlled residuals identify boundaries requiring further explanation. Other models, devices and complete searches need their own evidence.

See [JAX PSD guide](../../docs/jax_psd.rst) and [PSD numerical differences](../../docs/jax_psd_numerical_differences.rst).

## A held-spectrum power rounding witness

These complex64 inputs fix the preceding Fourier transform. Rounding each square before adding is distinct from retaining the real square until addition. The comparison reports which control matches each displayed library rather than assuming which side uses fused arithmetic. The subsequent float32 reciprocal makes the same power boundary visible in inverse-ASD truncation. `abs(z)**2` is displayed as a separate reordered formula; it is not either implementation's source expression.


In [9]:
import jax
import platform
try:
    WITNESS_DEVICE = "cuda:0" if jax.devices("gpu") else DEVICE
except RuntimeError:
    WITNESS_DEVICE = DEVICE
with JAXScheme(WITNESS_DEVICE) as witness_context:
    print({"JAX": jax.__version__, "NumPy": np.__version__, "CPU architecture": platform.machine(), "witness platform": witness_context.jax_device.platform, "witness device": witness_context.jax_device.device_kind})
# Identical complex64 spectra isolate power rounding from the preceding FFT.
# These finite values make two mathematically equivalent expansions differ.
import jax
import jax.numpy as jnp

fixed_spectra = np.array([complex(6.472284439951181e-6, 5.454407073557377e-6),
                          complex(0.0728302001953125, -0.00025936285965144634)], dtype=np.complex64)
r, i = fixed_spectra.real, fixed_spectra.imag
rounded_r2, rounded_i2 = np.float32(r*r), np.float32(i*i)
separate_squares = np.float32(rounded_r2 + rounded_i2)
retained_real_square = np.float32(r.astype(np.float64)**2 + rounded_i2.astype(np.float64))
assert np.all(separate_squares != retained_real_square)
original_power = np.abs(fixed_spectra * fixed_spectra.conj())
with JAXScheme(WITNESS_DEVICE) as active:
    gpu_spectra = jax.device_put(fixed_spectra, active.jax_device)
    jax_power = np.array(jax.jit(lambda z: jnp.real(z*jnp.conj(z)))(gpu_spectra))
print('Native abs(z*conj(z)):', original_power)
print('JAX real(z*conj(z)):', jax_power)
print('Separate rounded squares:', separate_squares)
print('Retain the real square until addition:', retained_real_square)
print('Native matches retained-real-square control:', original_power.tobytes() == retained_real_square.tobytes())
print('JAX matches separate-squares control:', jax_power.tobytes() == separate_squares.tobytes())
# Inverse-ASD truncation applies a reciprocal after this same power boundary.
# A one-step power change can produce a different float32 PSD reciprocal.
print('Reciprocals of the two expansions:', np.float32(1)/separate_squares,
      np.float32(1)/retained_real_square)
assert (np.float32(1)/separate_squares).tobytes() != (np.float32(1)/retained_real_square).tobytes()
# Taking magnitude first introduces its own rounding; this alternative is
# intentionally displayed separately from either implementation's source.
print('Alternative abs(z)**2:', np.float32(np.abs(fixed_spectra)**2))


{'JAX': '0.11.2', 'NumPy': '2.2.5', 'CPU architecture': 'x86_64', 'witness platform': 'gpu', 'witness device': 'NVIDIA GeForce RTX 3090'}
Native abs(z*conj(z)): [7.1641020e-11 5.3043053e-03]
JAX real(z*conj(z)): [7.1641026e-11 5.3043049e-03]
Separate rounded squares: [7.1641026e-11 5.3043049e-03]
Retain the real square until addition: [7.1641020e-11 5.3043053e-03]
Native matches retained-real-square control: True
JAX matches separate-squares control: True
Reciprocals of the two expansions: [1.3958482e+10 1.8852612e+02] [1.3958484e+10 1.8852609e+02]
Alternative abs(z)**2: [7.1641026e-11 5.3043053e-03]
